# Car Market Trends Analysis

## Data Analytics Project Using CarDekho Used-Car Dataset

**Project objective:** Analyze factors influencing used-car selling prices, measure depreciation, compare market segments, and identify practical insights for pricing and purchasing decisions.

This notebook follows the reference project workflow:

**Load & Inspect → Clean & Validate → Feature Engineering → Exploratory Data Analysis → Visualization → Insights → Recommendations**

The source project describes the dataset as containing 301 CarDekho used-car records and focuses on pricing, depreciation, mileage, fuel type, transmission, seller type, and ownership. 

## 1. Import Required Libraries

We use:
- **Pandas** for data loading, cleaning, transformation, and analysis.
- **NumPy** for numerical operations.
- **Matplotlib** for charts and visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns when viewing DataFrames
pd.set_option("display.max_columns", None)

# Make plots readable
plt.rcParams["figure.figsize"] = (8, 5)

print("Libraries imported successfully.")

## 2. Load the Dataset

The CarDekho CSV is loaded into a Pandas DataFrame called `df`.

In [ ]:
# Load the CarDekho dataset
file_path = r"1776311302-P3-Car Market Trends Analysis with Car Dekho Data (1).csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

## 3. First Look at the Data

We inspect the first and last records to understand how the dataset is structured.

In [ ]:
display(df.head())
display(df.tail())

## 4. Dataset Structure

The original dataset contains 9 columns covering:
- Car name
- Manufacturing year
- Selling price
- Present price
- Kilometers driven
- Fuel type
- Seller type
- Transmission
- Owner category

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

## 5. Statistical Summary

`describe()` gives basic statistics for numerical variables such as mean, minimum, maximum, and quartiles.

In [ ]:
df.describe(include="all").T

## 6. Data Quality Check

Before analysis, we check:
1. Missing values
2. Duplicate records
3. Data types

In [ ]:
print("Missing values by column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

## 7. Data Cleaning

We remove exact duplicate rows. Missing values are not filled artificially; if any are present, they should be handled according to the affected variable and analytical requirement.

The reference project uses data preparation and validation before analysis.

In [ ]:
# Store source-row count for reporting
source_rows = len(df)

# Remove exact duplicate records
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("Source rows:", source_rows)
print("Duplicates removed:", duplicate_count)
print("Rows after cleaning:", len(df))

print("\nMissing values after cleaning:")
display(df.isnull().sum().to_frame("Missing Values"))

## 8. Feature Engineering

The reference project creates:
- **Car Age**, using 2020 as the reference year.
- **Depreciation Percentage**
- **Brand**, derived from the car name for brand analysis.

In [ ]:
# Create car age using 2020 as the reference year
df["Car_Age"] = 2020 - df["Year"]

# Calculate depreciation amount
df["Depreciation"] = df["Present_Price"] - df["Selling_Price"]

# Calculate depreciation percentage
df["Depreciation_Pct"] = (
    (df["Present_Price"] - df["Selling_Price"])
    / df["Present_Price"]
) * 100

# Extract brand from the first word of the car name
df["Brand"] = (
    df["Car_Name"]
    .astype(str)
    .str.split()
    .str[0]
    .str.title()
)

display(df.head())

## 9. Key Performance Indicators (KPIs)

We calculate the main project KPIs:
- Number of cars analyzed
- Average selling price
- Highest selling price
- Average depreciation

In [ ]:
total_cars = len(df)
average_selling_price = df["Selling_Price"].mean()
highest_selling_price = df["Selling_Price"].max()
average_depreciation = df["Depreciation_Pct"].mean()

kpi = pd.DataFrame({
    "KPI": [
        "Cars Analyzed",
        "Average Selling Price",
        "Highest Selling Price",
        "Average Depreciation"
    ],
    "Value": [
        total_cars,
        round(average_selling_price, 2),
        round(highest_selling_price, 2),
        f"{average_depreciation:.2f}%"
    ]
})

display(kpi)

## 10. Present Price vs Selling Price

This analysis examines whether a car's present/reference price is associated with its used-car selling price.

In [ ]:
price_correlation = df["Present_Price"].corr(df["Selling_Price"])

plt.figure()
plt.scatter(df["Present_Price"], df["Selling_Price"], alpha=0.6)
plt.xlabel("Present Price")
plt.ylabel("Selling Price")
plt.title("Present Price vs Selling Price")
plt.tight_layout()
plt.show()

print(f"Correlation between Present Price and Selling Price: {price_correlation:.3f}")

## 11. Kilometers Driven vs Selling Price

Mileage is analyzed to see whether the distance driven is linearly associated with resale price.

In [ ]:
kms_correlation = df["Kms_Driven"].corr(df["Selling_Price"])

plt.figure()
plt.scatter(df["Kms_Driven"], df["Selling_Price"], alpha=0.6)
plt.xlabel("Kilometers Driven")
plt.ylabel("Selling Price")
plt.title("Kilometers Driven vs Selling Price")
plt.tight_layout()
plt.show()

print(f"Correlation between Kilometers Driven and Selling Price: {kms_correlation:.3f}")

## 12. Car Age vs Selling Price

Car age is analyzed because older vehicles may have lower resale values.

In [ ]:
age_correlation = df["Car_Age"].corr(df["Selling_Price"])

plt.figure()
plt.scatter(df["Car_Age"], df["Selling_Price"], alpha=0.6)
plt.xlabel("Car Age (Years)")
plt.ylabel("Selling Price")
plt.title("Car Age vs Selling Price")
plt.tight_layout()
plt.show()

print(f"Correlation between Car Age and Selling Price: {age_correlation:.3f}")

## 13. Average Selling Price by Fuel Type

We compare average selling prices across fuel categories.

In [ ]:
fuel_analysis = (
    df.groupby("Fuel_Type")
      .agg(
          Records=("Selling_Price", "size"),
          Average_Selling_Price=("Selling_Price", "mean"),
          Average_Depreciation=("Depreciation_Pct", "mean")
      )
      .sort_values("Average_Selling_Price", ascending=False)
)

display(fuel_analysis.round(2))

plt.figure()
plt.bar(
    fuel_analysis.index.astype(str),
    fuel_analysis["Average_Selling_Price"]
)
plt.xlabel("Fuel Type")
plt.ylabel("Average Selling Price")
plt.title("Average Selling Price by Fuel Type")
plt.tight_layout()
plt.show()

## 14. Transmission Analysis

We compare average selling prices for manual and automatic vehicles.

In [ ]:
transmission_analysis = (
    df.groupby("Transmission")
      .agg(
          Records=("Selling_Price", "size"),
          Average_Selling_Price=("Selling_Price", "mean")
      )
      .sort_values("Average_Selling_Price", ascending=False)
)

display(transmission_analysis.round(2))

plt.figure()
plt.bar(
    transmission_analysis.index.astype(str),
    transmission_analysis["Average_Selling_Price"]
)
plt.xlabel("Transmission")
plt.ylabel("Average Selling Price")
plt.title("Average Selling Price by Transmission")
plt.tight_layout()
plt.show()

## 15. Seller Type Analysis

This compares the average selling price between seller categories.

In [ ]:
seller_analysis = (
    df.groupby("Seller_Type")
      .agg(
          Records=("Selling_Price", "size"),
          Average_Selling_Price=("Selling_Price", "mean")
      )
      .sort_values("Average_Selling_Price", ascending=False)
)

display(seller_analysis.round(2))

plt.figure()
plt.bar(
    seller_analysis.index.astype(str),
    seller_analysis["Average_Selling_Price"]
)
plt.xlabel("Seller Type")
plt.ylabel("Average Selling Price")
plt.title("Average Selling Price by Seller Type")
plt.tight_layout()
plt.show()

## 16. Owner Analysis

We compare average selling prices across previous-owner categories.

In [ ]:
owner_analysis = (
    df.groupby("Owner")
      .agg(
          Records=("Selling_Price", "size"),
          Average_Selling_Price=("Selling_Price", "mean"),
          Average_Depreciation=("Depreciation_Pct", "mean")
      )
      .sort_values("Average_Selling_Price", ascending=False)
)

display(owner_analysis.round(2))

plt.figure()
plt.bar(
    owner_analysis.index.astype(str),
    owner_analysis["Average_Selling_Price"]
)
plt.xlabel("Owner Category")
plt.ylabel("Average Selling Price")
plt.title("Average Selling Price by Owner Category")
plt.tight_layout()
plt.show()

## 17. Brand Analysis

The first word of each car name is used as a simple brand identifier. To reduce distortion from brands with very few observations, we display the top brands among those with at least 3 records.

In [ ]:
brand_analysis = (
    df.groupby("Brand")
      .agg(
          Records=("Selling_Price", "size"),
          Average_Selling_Price=("Selling_Price", "mean")
      )
)

brand_filtered = (
    brand_analysis[brand_analysis["Records"] >= 3]
    .sort_values("Average_Selling_Price", ascending=False)
)

display(brand_filtered.head(15).round(2))

top_brands = brand_filtered.head(10).sort_values(
    "Average_Selling_Price"
)

plt.figure(figsize=(9, 6))
plt.barh(
    top_brands.index,
    top_brands["Average_Selling_Price"]
)
plt.xlabel("Average Selling Price")
plt.ylabel("Brand")
plt.title("Top Brands by Average Selling Price (Minimum 3 Records)")
plt.tight_layout()
plt.show()

## 18. Depreciation Analysis

Depreciation measures the reduction from present price to selling price.

**Formula:**

`Depreciation % = ((Present Price - Selling Price) / Present Price) × 100`

In [ ]:
display(
    df[
        ["Car_Name", "Present_Price", "Selling_Price",
         "Depreciation", "Depreciation_Pct"]
    ].head(10).round(2)
)

plt.figure()
plt.hist(df["Depreciation_Pct"], bins=20)
plt.xlabel("Depreciation (%)")
plt.ylabel("Number of Cars")
plt.title("Distribution of Car Depreciation")
plt.tight_layout()
plt.show()

## 19. Correlation Analysis

Correlation helps identify linear relationships between selling price and selected numerical variables.

**Important:** correlation indicates association, not causation.

In [ ]:
correlation_data = df[
    [
        "Selling_Price",
        "Present_Price",
        "Kms_Driven",
        "Car_Age",
        "Depreciation_Pct"
    ]
]

correlation_matrix = correlation_data.corr()

display(correlation_matrix.round(3))

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(correlation_matrix, interpolation="nearest")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

plt.colorbar(label="Correlation")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 20. Key Insights

The following statements are generated from the calculations above. They should be read together with the sample sizes of each category.

In [ ]:
print("KEY INSIGHTS")
print("-" * 60)
print(f"1. Average selling price: {average_selling_price:.2f}")
print(f"2. Highest selling price: {highest_selling_price:.2f}")
print(f"3. Average depreciation: {average_depreciation:.2f}%")
print(f"4. Present Price vs Selling Price correlation: {price_correlation:.3f}")
print(f"5. Car Age vs Selling Price correlation: {age_correlation:.3f}")
print(f"6. Kilometers Driven vs Selling Price correlation: {kms_correlation:.3f}")

highest_fuel = fuel_analysis["Average_Selling_Price"].idxmax()
highest_transmission = transmission_analysis["Average_Selling_Price"].idxmax()
highest_seller = seller_analysis["Average_Selling_Price"].idxmax()

print(f"7. Highest average selling price by fuel type: {highest_fuel}")
print(f"8. Highest average selling price by transmission: {highest_transmission}")
print(f"9. Highest average selling price by seller type: {highest_seller}")

## 21. Business Recommendations

Based on the analysis:
- Use present price, car age, and mileage together when evaluating resale value.
- Compare cars within similar fuel and transmission segments.
- Prefer newer vehicles when resale potential is an important consideration.
- Use historical depreciation patterns to support purchase and inventory decisions.
- Treat very small categories cautiously and validate conclusions with additional data.

## 22. Final Analytical Summary

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Cars analyzed",
        "Average selling price",
        "Highest selling price",
        "Average depreciation (%)",
        "Present price correlation",
        "Car age correlation",
        "Mileage correlation"
    ],
    "Value": [
        total_cars,
        round(average_selling_price, 2),
        round(highest_selling_price, 2),
        round(average_depreciation, 2),
        round(price_correlation, 3),
        round(age_correlation, 3),
        round(kms_correlation, 3)
    ]
})

display(summary)

## 23. Export the Cleaned Dataset

The final dataset includes the original variables plus `Car_Age`, `Depreciation`, `Depreciation_Pct`, and `Brand`.

In [ ]:
output_file = "CarDekho_Cleaned_Data.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")

## 24. Conclusion

The analysis identifies pricing and depreciation patterns in the CarDekho used-car dataset. Present price, car age, mileage, fuel type, transmission, seller type, ownership, and brand can be examined as resale-value indicators.

The project can be extended into an interactive dashboard, such as a Power BI dashboard, after the notebook analysis is finalized.